# AudioGen Batch Synthesis Notebook
Headless batch synthesis for Hindi and Punjabi reel scripts.

In [ ]:
MANIFEST_PATH = "scripts.json"
MODEL_WEIGHTS_DIR = "/kaggle/input/audiogen-weights"
OUTPUT_DIR = "outputs"


In [ ]:
import sys
import os
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("audiogen_batch")

# Add project roots to sys.path
for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/working/audiogen")]:
    src_dir = candidate / "src"
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    if (candidate / "voices").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
    if (candidate / "batch").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from audiogen.engine import Synthesizer
from audiogen.audio_processor import AudioProcessor
from voices.registry import get_voice_ref
from batch.manifest_schema import BatchJob, ReelAudioTask


In [ ]:
manifest_file = Path(MANIFEST_PATH)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

with open(manifest_file, "r", encoding="utf-8") as f:
    raw_manifest = json.load(f)

if isinstance(raw_manifest, list):
    batch_job = BatchJob(tasks=raw_manifest)
elif isinstance(raw_manifest, dict) and "tasks" in raw_manifest:
    batch_job = BatchJob(**raw_manifest)
else:
    raise ValueError(f"Invalid manifest structure in {manifest_file}")

logger.info(f"Loaded {len(batch_job.tasks)} task(s) for batch synthesis.")

# Model loaded ONCE (warm process)
weights_path = Path(MODEL_WEIGHTS_DIR) if MODEL_WEIGHTS_DIR else None
if weights_path is not None and not weights_path.exists():
    logger.warning(f"Weights path {weights_path} not found; falling back to environment/local.")
    weights_path = None

if weights_path and weights_path.exists():
    synthesizer = Synthesizer(model_path=weights_path)
else:
    try:
        synthesizer = Synthesizer()
    except Exception:
        synthesizer = Synthesizer(model_path=Path.cwd())

processor = AudioProcessor(target_sample_rate=24000)
logger.info(f"Initialized Synthesizer on device: {synthesizer.device}")


In [ ]:
results = []
for task in batch_job.tasks:
    task_id = task.id
    logger.info(f"Processing task '{task_id}' (language={task.language}, voice_ref={task.voice_ref})")
    try:
        ref_audio_path = get_voice_ref(task.voice_ref)
        raw_audio, sr = synthesizer.synthesize(
            text=task.text,
            language=task.language,
            speaker_ref=ref_audio_path,
        )
        out_wav_path = output_dir / f"{task_id}.wav"
        processor.process_and_export(raw_audio, sr, out_wav_path)
        results.append({
            "id": task_id,
            "status": "success",
            "output_file": str(out_wav_path),
            "error": None
        })
        logger.info(f"Task '{task_id}' completed successfully.")
    except Exception as exc:
        logger.error(f"Task '{task_id}' failed: {exc}", exc_info=True)
        results.append({
            "id": task_id,
            "status": "failed",
            "output_file": None,
            "error": str(exc)
        })

manifest_output_path = output_dir / "manifest_output.json"
summary_data = {
    "total": len(results),
    "successful": sum(1 for r in results if r["status"] == "success"),
    "failed": sum(1 for r in results if r["status"] == "failed"),
    "tasks": results
}
with open(manifest_output_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

logger.info(f"Manifest output written to {manifest_output_path}")


In [ ]:
sys.exit(0)
